# Notebook 2: Human protein GO term prediction

From notebook 1 we found dropout=0.1 worked best on the simulated data.
Here we apply the same CNN to real human proteins and predict 5 GO terms.
We also try ESM-2, a pretrained protein language model, following the project handout.
We test two model sizes (8M and 35M) and combine the best into an ensemble.

In [ ]:
import os
import re
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import matplotlib.pyplot as plt

if not os.path.exists("grs34806-deep-learning-project-data"):
    !git clone https://git.wur.nl/bioinformatics/grs34806-deep-learning-project-data.git

if os.path.basename(os.getcwd()) != "grs34806-deep-learning-project-data":
    os.chdir("grs34806-deep-learning-project-data")

## Load data

Same regex trick as notebook 1 — id and sequence are sometimes on the same line.

In [ ]:
seqfile = "expr5Tseq_filtGO_100-1000.lis"

ids, seqs = [], []
with open(seqfile) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if "\t" in line:
            pid, s = line.split("\t", 1)
        else:
            m = re.match(r"^([^A-Z]+)([A-Z].*)$", line)
            if not m:
                continue
            pid, s = m.group(1), m.group(2)
        ids.append(pid.strip())
        seqs.append(s.strip())

print(len(seqs), "sequences loaded")

In [ ]:
# 5 GO term annotation files
go_terms = [
    ("GO_3A0005739.annotprot", "mitochondrion"),
    ("GO_3A0007165.annotprot", "signal_transduction"),
    ("GO_3A0005576.annotprot", "extracellular"),
    ("GO_3A0055085.annotprot", "transmembrane"),
    ("GO_3A0043066.annotprot", "neg_apoptosis"),
]

id2idx = {pid: i for i, pid in enumerate(ids)}
labels = np.zeros((len(ids), len(go_terms)), dtype=np.float32)

for col, (fname, name) in enumerate(go_terms):
    with open(fname) as f:
        pos_ids = [l.strip() for l in f if l.strip()]
    for pid in pos_ids:
        if pid in id2idx:
            labels[id2idx[pid], col] = 1.0
    print(name, int(labels[:, col].sum()))

labels = torch.tensor(labels)

## Tokenise (same functions as notebook 1 and the handout)

In [ ]:
mapaa2num = {aa: i for (i, aa) in enumerate(list("ACDEFGHIKLMNPQRSTVWY"))}

def tokenize(dat, map2num, non_aa_num=20):
    seq = []
    for count, i in enumerate(dat):
        seq.append([map2num.get(j, non_aa_num) for j in list(i)])
    return seq

def truncate_pad(line, num_steps, padding_token):
    if len(line) > num_steps:
        return line[:num_steps]
    return line + [padding_token] * (num_steps - len(line))

def build_seq_array(lines, num_steps, non_aa_num=20):
    array = torch.tensor([truncate_pad(l, num_steps, non_aa_num) for l in lines])
    return array

num_steps = 1000
X = build_seq_array(tokenize(seqs, mapaa2num), num_steps)
X.shape

In [ ]:
# 70/15/15 split — save indices so CNN and ESM-2 use the same split (fair comparison)
torch.manual_seed(42)
perm    = torch.randperm(len(X))
n_train = int(0.7  * len(X))
n_val   = int(0.15 * len(X))

train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train + n_val]
test_idx  = perm[n_train + n_val:]

X_train, y_train = X[train_idx], labels[train_idx]
X_val,   y_val   = X[val_idx],   labels[val_idx]
X_test,  y_test  = X[test_idx],  labels[test_idx]

print(f"train {len(X_train)}  val {len(X_val)}  test {len(X_test)}")

batch_size = 32
train_iter = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_iter   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=batch_size)
test_iter  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=batch_size)

## 1. CNN

Same 2-block embedding CNN as notebook 1 but with 5 output labels instead of 1.
Uses dropout=0.1 from the notebook 1 hyperparameter sweep.
BCEWithLogitsLoss with pos_weight because the classes are very imbalanced (3-7% positives).

In [ ]:
class ProteinCNN(nn.Module):
    def __init__(self, vocab_size=21, conv_channels=64, dropout=0.1, n_labels=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, 64, padding_idx=vocab_size)
        self.cnn = nn.Sequential(
            # conv block 1
            nn.Conv1d(64, conv_channels, kernel_size=8, padding=4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(2, 2),
            # conv block 2
            nn.Conv1d(conv_channels, conv_channels, kernel_size=8, padding=4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.AdaptiveMaxPool1d(1),
            nn.Flatten(),
            nn.Linear(conv_channels, n_labels),
        )

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        return self.cnn(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# weight positives more — stops the model from just predicting all negatives
pos        = y_train.sum(0)
neg        = y_train.size(0) - pos
pos_weight = (neg / pos.clamp(min=1)).to(device)

def eval_auprc(net, loader):
    net.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            all_p.append(net(xb.to(device)).cpu())
            all_y.append(yb)
    probs = torch.sigmoid(torch.cat(all_p)).numpy()
    y = torch.cat(all_y).numpy()
    aps = [average_precision_score(y[:, i], probs[:, i]) for i in range(y.shape[1]) if y[:, i].sum() > 0]
    return float(np.mean(aps))

def get_probs(net, loader):
    net.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            all_p.append(net(xb.to(device)).cpu())
            all_y.append(yb)
    return torch.sigmoid(torch.cat(all_p)).numpy(), torch.cat(all_y).numpy()

In [ ]:
torch.manual_seed(0)
net       = ProteinCNN(dropout=0.1, n_labels=labels.shape[1]).to(device)
loss_fn   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

best_val, best_state, no_improve, patience = -1, None, 0, 5

for epoch in range(1, 26):
    net.train()
    total = 0.0
    for xb, yb in train_iter:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(net(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
    val_auprc = eval_auprc(net, val_iter)
    if val_auprc > best_val:
        best_val, best_state, no_improve = val_auprc, copy.deepcopy(net.state_dict()), 0
    else:
        no_improve += 1
    print(f"epoch {epoch:2d}  loss {total/len(train_iter.dataset):.3f}  val mAUPRC {val_auprc:.3f}")
    if no_improve >= patience:
        break

net.load_state_dict(best_state)
cnn_probs, y_true = get_probs(net, test_iter)

print("\nCNN test results:")
for i, (_, name) in enumerate(go_terms):
    auroc = roc_auc_score(y_true[:, i], cnn_probs[:, i])
    auprc = average_precision_score(y_true[:, i], cnn_probs[:, i])
    print(f"{name}  AUROC {auroc:.3f}  AUPRC {auprc:.3f}")

## 2. ESM-2 (pretrained protein language model)

ESM-2 was pretrained on 65M protein sequences so it already knows about protein biology.
We freeze its weights and train only a small MLP head on top of the embeddings.
We test two sizes: 8M params (6 layers) and 35M params (12 layers).
This follows the skeleton in the project handout.

In [ ]:
!pip install transformers -q
from transformers import AutoTokenizer, AutoModel

# preprocessing helpers from the project handout
context_size = 1000

def pad_or_trim(seq, size):
    if len(seq) > size:
        return seq[:size]
    else:
        return seq + '_' * (size - len(seq))

def add_spaces_between_characters(seq):
    return ''.join(f'{s} ' for s in seq)[:-1]

def prepare_seq(seq):
    seq = pad_or_trim(seq, context_size)
    seq = add_spaces_between_characters(seq)
    seq = re.sub(r"[UZOB]", "X", seq)
    return seq

### ESM-2 8M

In [ ]:
esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
base_model    = AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D")

for parameter in base_model.parameters():
    parameter.requires_grad = False

base_model = base_model.to(device)
base_model.eval()
print("embedding dim:", base_model.config.hidden_size)

In [ ]:
# run every sequence through ESM-2, mean-pool to get one vector per protein
emb_batch  = 16
embeddings = []

with torch.no_grad():
    for i in range(0, len(seqs), emb_batch):
        batch  = [prepare_seq(s) for s in seqs[i:i+emb_batch]]
        tokens = esm_tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        tokens = {k: v.to(device) for k, v in tokens.items()}
        out    = base_model(**tokens).last_hidden_state
        embeddings.append(out.mean(dim=1).cpu())
        if i % 800 == 0:
            print(i)

embeddings = torch.cat(embeddings, dim=0)
embeddings.shape

In [ ]:
emb_dim = base_model.config.hidden_size

E_train = embeddings[train_idx]
E_val   = embeddings[val_idx]
E_test  = embeddings[test_idx]

esm_train_iter = DataLoader(TensorDataset(E_train, y_train), batch_size=64, shuffle=True)
esm_val_iter   = DataLoader(TensorDataset(E_val,   y_val),   batch_size=64)
esm_test_iter  = DataLoader(TensorDataset(E_test,  y_test),  batch_size=64)

class ESMHead(nn.Module):
    def __init__(self, emb_dim, n_labels=5, dropout=0.1):
        super().__init__()
        self.fc1     = nn.Linear(emb_dim, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(128, n_labels)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))

head_8m    = ESMHead(emb_dim=emb_dim, n_labels=labels.shape[1], dropout=0.1).to(device)
head_loss  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
head_opt   = torch.optim.AdamW(head_8m.parameters(), lr=1e-3, weight_decay=1e-2)

best_val, best_state, no_improve, patience = -1, None, 0, 15

for epoch in range(1, 101):
    head_8m.train()
    total = 0.0
    for xb, yb in esm_train_iter:
        xb, yb = xb.to(device), yb.to(device)
        loss = head_loss(head_8m(xb), yb)
        head_opt.zero_grad()
        loss.backward()
        head_opt.step()
        total += loss.item() * xb.size(0)
    val_auprc = eval_auprc(head_8m, esm_val_iter)
    if val_auprc > best_val:
        best_val, best_state, no_improve = val_auprc, copy.deepcopy(head_8m.state_dict()), 0
    else:
        no_improve += 1
    print(f"epoch {epoch:2d}  loss {total/len(esm_train_iter.dataset):.3f}  val mAUPRC {val_auprc:.3f}")
    if no_improve >= patience:
        break

head_8m.load_state_dict(best_state)
esm8m_probs, _ = get_probs(head_8m, esm_test_iter)

print("\nESM-2 8M test results:")
for i, (_, name) in enumerate(go_terms):
    auroc = roc_auc_score(y_true[:, i], esm8m_probs[:, i])
    auprc = average_precision_score(y_true[:, i], esm8m_probs[:, i])
    print(f"{name}  AUROC {auroc:.3f}  AUPRC {auprc:.3f}")

### ESM-2 35M

Larger model — free the 8M from GPU first to avoid running out of memory.

In [ ]:
del base_model
torch.cuda.empty_cache()

esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
base_model    = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D")

for parameter in base_model.parameters():
    parameter.requires_grad = False

base_model = base_model.to(device)
base_model.eval()
print("embedding dim:", base_model.config.hidden_size)

In [ ]:
emb_batch     = 8  # smaller batch to fit the bigger model in memory
embeddings_35 = []

with torch.no_grad():
    for i in range(0, len(seqs), emb_batch):
        batch  = [prepare_seq(s) for s in seqs[i:i+emb_batch]]
        tokens = esm_tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        tokens = {k: v.to(device) for k, v in tokens.items()}
        out    = base_model(**tokens).last_hidden_state
        embeddings_35.append(out.mean(dim=1).cpu())
        if i % 800 == 0:
            print(i)

embeddings_35 = torch.cat(embeddings_35, dim=0)
embeddings_35.shape

In [ ]:
emb_dim_35 = base_model.config.hidden_size

E_train_35 = embeddings_35[train_idx]
E_val_35   = embeddings_35[val_idx]
E_test_35  = embeddings_35[test_idx]

esm35_train_iter = DataLoader(TensorDataset(E_train_35, y_train), batch_size=64, shuffle=True)
esm35_val_iter   = DataLoader(TensorDataset(E_val_35,   y_val),   batch_size=64)
esm35_test_iter  = DataLoader(TensorDataset(E_test_35,  y_test),  batch_size=64)

head_35m   = ESMHead(emb_dim=emb_dim_35, n_labels=labels.shape[1], dropout=0.1).to(device)
head_opt35 = torch.optim.AdamW(head_35m.parameters(), lr=1e-3, weight_decay=1e-2)

best_val, best_state, no_improve, patience = -1, None, 0, 15

for epoch in range(1, 101):
    head_35m.train()
    total = 0.0
    for xb, yb in esm35_train_iter:
        xb, yb = xb.to(device), yb.to(device)
        loss = head_loss(head_35m(xb), yb)
        head_opt35.zero_grad()
        loss.backward()
        head_opt35.step()
        total += loss.item() * xb.size(0)
    val_auprc = eval_auprc(head_35m, esm35_val_iter)
    if val_auprc > best_val:
        best_val, best_state, no_improve = val_auprc, copy.deepcopy(head_35m.state_dict()), 0
    else:
        no_improve += 1
    print(f"epoch {epoch:2d}  loss {total/len(esm35_train_iter.dataset):.3f}  val mAUPRC {val_auprc:.3f}")
    if no_improve >= patience:
        break

head_35m.load_state_dict(best_state)
esm35m_probs, _ = get_probs(head_35m, esm35_test_iter)

print("\nESM-2 35M test results:")
for i, (_, name) in enumerate(go_terms):
    auroc = roc_auc_score(y_true[:, i], esm35m_probs[:, i])
    auprc = average_precision_score(y_true[:, i], esm35m_probs[:, i])
    print(f"{name}  AUROC {auroc:.3f}  AUPRC {auprc:.3f}")

## Model comparison

In [ ]:
print("AUROC comparison:")
for i, (_, name) in enumerate(go_terms):
    c   = roc_auc_score(y_true[:, i], cnn_probs[:, i])
    m8  = roc_auc_score(y_true[:, i], esm8m_probs[:, i])
    m35 = roc_auc_score(y_true[:, i], esm35m_probs[:, i])
    print(f"{name}  CNN {c:.3f}  8M {m8:.3f}  35M {m35:.3f}")

## Ensemble: CNN + ESM-2 35M

CNN is stronger on motif-heavy terms (extracellular, transmembrane).
ESM-2 35M is stronger on broad terms (signal transduction, neg apoptosis).
Averaging their probabilities combines both.

In [ ]:
ens_probs = (cnn_probs + esm35m_probs) / 2

print("Ensemble test results:")
for i, (_, name) in enumerate(go_terms):
    auroc = roc_auc_score(y_true[:, i], ens_probs[:, i])
    auprc = average_precision_score(y_true[:, i], ens_probs[:, i])
    print(f"{name}  AUROC {auroc:.3f}  AUPRC {auprc:.3f}")

# bar chart comparing AUROC across all models
names = [n for _, n in go_terms]
x = np.arange(len(names))
w = 0.2
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w*1.5, [roc_auc_score(y_true[:,i], cnn_probs[:,i])    for i in range(5)], w, label="CNN")
ax.bar(x - w*0.5, [roc_auc_score(y_true[:,i], esm8m_probs[:,i])  for i in range(5)], w, label="ESM-2 8M")
ax.bar(x + w*0.5, [roc_auc_score(y_true[:,i], esm35m_probs[:,i]) for i in range(5)], w, label="ESM-2 35M")
ax.bar(x + w*1.5, [roc_auc_score(y_true[:,i], ens_probs[:,i])    for i in range(5)], w, label="Ensemble")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("AUROC")
ax.set_title("Test AUROC per GO term")
ax.legend()
plt.tight_layout()
plt.savefig("auroc_comparison.png", dpi=150)
plt.show()

## Predictions on the unlabeled test set (14,765 proteins)

We use ESM-2 35M (best single model) for the final predictions.
Threshold is tuned per GO term using validation set F1 —
needed because pos_weight shifts the model's raw outputs above 0.5.

In [ ]:
!pip install biopython -q
from Bio import SeqIO

test_seqs, test_ids = [], []
for rec in SeqIO.parse("test_set_filt.f", "fasta"):
    test_ids.append(str(rec.id))
    test_seqs.append(str(rec.seq))

print(len(test_seqs), "proteins")

In [ ]:
# embed the unlabeled test proteins the same way as the training ones
test_embeddings = []
with torch.no_grad():
    for i in range(0, len(test_seqs), emb_batch):
        batch  = [prepare_seq(s) for s in test_seqs[i:i+emb_batch]]
        tokens = esm_tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        tokens = {k: v.to(device) for k, v in tokens.items()}
        out    = base_model(**tokens).last_hidden_state
        test_embeddings.append(out.mean(dim=1).cpu())
        if i % 1600 == 0:
            print(i)

test_embeddings = torch.cat(test_embeddings, dim=0)
test_embeddings.shape

In [ ]:
# find the threshold per GO term that maximises F1 on the validation set
val_logits = head_35m(E_val_35.to(device)).detach().cpu().numpy()
val_probs  = 1 / (1 + np.exp(-val_logits))

print("Optimal thresholds:")
thresholds = {}
for i, (_, name) in enumerate(go_terms):
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.05, 0.95, 0.05):
        f1 = f1_score(y_val.numpy()[:, i],
                      (val_probs[:, i] > t).astype(int),
                      zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds[name] = best_t
    print(f"{name}  threshold {best_t:.2f}  val F1 {best_f1:.3f}")

# apply to test set
head_35m.eval()
with torch.no_grad():
    test_logits = head_35m(test_embeddings.to(device)).cpu().numpy()
test_probs = 1 / (1 + np.exp(-test_logits))

print("\nPredicted positives per GO term:")
for i, (_, name) in enumerate(go_terms):
    n = int((test_probs[:, i] > thresholds[name]).sum())
    print(f"{name}  {n}  ({100*n/len(test_seqs):.1f}%)")